# Evaluation

The metrics and baselines are in the introduction. This page tunes the baselines on validation, scores everything on test once, and checks whether our gain over the best baseline is real and where the model fails. For validation, models are fit on train. For test, they are fit on train plus validation.

In [1]:
from pathlib import Path

import hvplot.polars
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression

from utils import LABELS

pl.Config.set_tbl_rows(30)
pl.Config.set_float_precision(3)
pl.Config.set_fmt_str_lengths(60)
DATA = Path("data")
SEED = 0
SPLITS = ["train", "val", "test"]
X = pl.read_parquet(DATA / "cache" / "final_eval.parquet")
tr, va, te = (X.filter(pl.col("split") == s) for s in SPLITS)
y_tr, y_va, y_te = (d["price"].to_numpy() for d in (tr, va, te))

---

## Metrics

Each metric favors a different constant guess: the median for MAE, the mean for MSE and the geometric mean for MSLE. Here they are on validation. A model should beat the constant that suits its own metric, not just one of them.

We report all three because they answer different questions. MAE is the typical dollar miss. MSE punishes the largest misses, and the table in Where it fails shows that \$30 games and up carry 65% of it. MSLE counts a \$20 miss on a free game as the large miss it is, which matters for a pricing decision at the free end.

In [2]:
def scores(y, p):
    p = np.clip(p, 0, None)
    return {"MAE": np.mean(np.abs(y - p)), "MSE": np.mean((y - p) ** 2),
            "MSLE": np.mean((np.log1p(y) - np.log1p(p)) ** 2)}


def table(preds, y):
    return pl.DataFrame([{"model": k, **scores(y, p)} for k, p in preds.items()])


constants = {"median": np.median(y_tr), "mean": y_tr.mean(), "geometric mean": np.expm1(np.log1p(y_tr).mean())}
table({f"constant: {k}": np.full(len(y_va), v) for k, v in constants.items()}, y_va)

model,MAE,MSE,MSLE
str,f64,f64,f64
"""constant: median""",12.673,319.589,1.809
"""constant: mean""",12.701,312.164,1.861
"""constant: geometric mean""",13.923,397.033,1.696


---

## Baselines

Two baselines have a setting, so we tune them on validation. In the recency-weighted mean, a higher alpha puts more weight on the latest review. The lowest alpha, 0.1, is best on all three metrics. The shrunk mean pulls a user's mean log price toward the overall mean by `n / (n + k)`, where `n` is the number of reviews they wrote. On validation, `k=5` is best on MSLE (`k=2` is almost tied), and `k=1` is best on MAE and MSE.

In [3]:
ewa = table({f"alpha={a}": va[f"ewm{a}"].to_numpy() for a in (0.1, 0.3, 0.5, 0.7, 0.9)}, y_va)


def shrunk(d, y_fit, k):
    g = np.log1p(y_fit).mean()
    n, m = d["n"].to_numpy(), d["m_lp"].to_numpy()
    return np.expm1(g + n / (n + k) * (m - g))


shrink = table({f"k={k}": shrunk(va, y_tr, k) for k in (0, 1, 2, 5, 10, 20, 50)}, y_va)
pl.concat([ewa.with_columns(family=pl.lit("recency-weighted mean")), shrink.with_columns(family=pl.lit("shrunk mean"))])

model,MAE,MSE,MSLE,family
str,f64,f64,f64,str
"""alpha=0.1""",12.948,375.114,1.955,"""recency-weighted mean"""
"""alpha=0.3""",13.091,379.543,1.971,"""recency-weighted mean"""
"""alpha=0.5""",13.357,391.849,2.004,"""recency-weighted mean"""
"""alpha=0.7""",13.758,413.662,2.070,"""recency-weighted mean"""
"""alpha=0.9""",14.316,449.038,2.225,"""recency-weighted mean"""
"""k=0""",13.228,396.432,1.957,"""shrunk mean"""
"""k=1""",12.635,348.518,1.577,"""shrunk mean"""
"""k=2""",12.795,355.974,1.521,"""shrunk mean"""
"""k=5""",13.124,368.758,1.520,"""shrunk mean"""


The price-bucket Markov chain puts the last price in a bucket (free, \$0-5, \$5-10 and so on) and predicts the median, mean or geometric mean of the next prices from that bucket. The game-level chain does the same with the last game itself. The linear regression uses the history features only. All baselines on validation:

In [4]:
ALPHA = 0.1
K_SHRINK = int(shrink.sort("MSLE")["model"][0].split("=")[1])
HIST = ["logn", "m_p", "m_lp", "free_sh", "sd_lp", "mx", "med10", "ewm0.1", "ewm_lp", "last_p", "last_lp"]

The seven price buckets used below: free, \$0-5, \$5-10, \$10-20, \$20-30, \$30-60 and \$60+.

In [5]:
def bucket_of(col):
    return pl.col(col).cut([0, 4.99, 9.99, 19.99, 29.99, 59.99], labels=LABELS).cast(pl.Enum(LABELS))

In [6]:
def baselines(d, fit):
    y_fit = fit["price"].to_numpy()
    chain = (fit.with_columns(state=bucket_of("last_p"))
                .group_by("state").agg(med=pl.col("price").median(), mean=pl.col("price").mean(),
                                       geo=pl.col("lp").mean().exp() - 1))
    state = d.with_columns(state=bucket_of("last_p")).join(chain, on="state", how="left")
    lin = LinearRegression().fit(fit.select(HIST).to_numpy(), np.log1p(y_fit))
    return {
        "constant: median": np.full(d.height, np.median(y_fit)),
        "constant: mean": np.full(d.height, y_fit.mean()),
        "constant: geometric mean": np.full(d.height, np.expm1(np.log1p(y_fit).mean())),
        "last price": d["last_p"].to_numpy(),
        "user median (last 10)": d["med10"].to_numpy(),
        f"recency-weighted mean (alpha={ALPHA})": d[f"ewm{ALPHA}"].to_numpy(),
        f"shrunk user mean (k={K_SHRINK})": shrunk(d, y_fit, K_SHRINK),
        "Markov chain on price buckets: median": state["med"].to_numpy(),
        "Markov chain on price buckets: mean": state["mean"].to_numpy(),
        "Markov chain on price buckets: geometric mean": state["geo"].to_numpy(),
        "Markov chain on games": np.expm1(d["tr_lp"].to_numpy()),
        "linear regression on history": np.expm1(lin.predict(d.select(HIST).to_numpy())),
    }

In [7]:
base_val = table(baselines(va, tr), y_va)
base_val

model,MAE,MSE,MSLE
str,f64,f64,f64
"""constant: median""",12.673,319.589,1.809
"""constant: mean""",12.701,312.164,1.861
"""constant: geometric mean""",13.923,397.033,1.696
"""last price""",14.542,465.547,2.462
"""user median (last 10)""",12.952,381.995,2.037
"""recency-weighted mean (alpha=0.1)""",12.948,375.114,1.955
"""shrunk user mean (k=5)""",13.124,368.758,1.520
"""Markov chain on price buckets: median""",12.235,311.857,1.670
"""Markov chain on price buckets: mean""",12.267,296.211,1.782


The Markov chain only helps if the next price depends on the last one. This table looks at training reviews: each row is the bucket of the user's last game, and the columns show what their next review cost.

In [8]:
regress = (tr.with_columns(last=bucket_of("last_p")).group_by("last")
             .agg(reviews=pl.len(), next_median=pl.col("price").median(), next_mean=pl.col("price").mean(),
                  percent_free=100 * (pl.col("price") == 0).mean())
             .sort("last"))
regress

last,reviews,next_median,next_mean,percent_free
enum,u32,f64,f64,f64
"""free""",159473,9.990,12.556,35.523
"""$0-5""",94037,9.990,12.604,13.924
"""$5-10""",178239,9.990,14.633,12.520
"""$10-20""",336240,14.990,17.188,12.561
"""$20-30""",103479,19.990,19.681,11.830
"""$30-60""",84977,19.990,21.150,12.407
"""$60+""",2892,14.990,17.133,17.289


We refit the baselines on train plus validation and score them next to our models. The weakest baseline on every metric is the last price. A user's last price is noisy, but the average of what people review after a game in the same bucket is the strongest baseline for MAE and MSE. That is regression to the mean, and the table above shows it: after a game priced \$60 or more, the next review averages \$17.13, against \$21.15 after a \$30-60 game. On MAE, the constant median (12.16) beats every history baseline except the bucket chain's median and mean versions.

In [9]:
full = pl.concat([tr, va])
base_pred = baselines(te, full)
MODELS = {"ridge, all features": "p_ridge", "GBM, log head": "p_plain_log", "GBM, absolute head": "p_plain_abs",
          "GBM, squared head": "p_plain_sq", "GBM + hurdle, log": "p_blend_log", "GBM + hurdle, squared": "p_blend_sq"}
test_pred = {**base_pred, **{name: te[col].to_numpy() for name, col in MODELS.items()}}
test_table = table(test_pred, y_te)
test_table

model,MAE,MSE,MSLE
str,f64,f64,f64
"""constant: median""",12.160,300.371,1.544
"""constant: mean""",12.189,289.245,1.596
"""constant: geometric mean""",13.734,382.032,1.517
"""last price""",14.917,462.716,2.458
"""user median (last 10)""",13.343,377.780,2.061
"""recency-weighted mean (alpha=0.1)""",13.279,368.237,1.942
"""shrunk user mean (k=5)""",13.179,358.551,1.408
"""Markov chain on price buckets: median""",11.974,294.240,1.458
"""Markov chain on price buckets: mean""",11.934,276.721,1.541


Our pick for each metric, chosen on validation, against the best baseline for that metric. The gain over the best baseline is 3.9% on MAE, 8.0% on MSE and 6.3% on MSLE. Against the recency-weighted mean, which the introduction lists as a baseline, the gain is 13.6%, 30.8% and 32.1%. The best baseline is picked from the test table. That choice favors the baselines, so the gains above are conservative.

In [10]:
ours = {m: te[f"p_ours_{m}"].to_numpy() for m in ("MAE", "MSE", "MSLE")}
baseline_names = list(base_pred)
rows = []
for m, p in ours.items():
    best = test_table.filter(pl.col("model").is_in(baseline_names)).sort(m).row(0, named=True)
    rows.append({"metric": m, "ours": scores(y_te, p)[m], "best baseline": best["model"], "baseline": best[m],
                 "improvement_pct": 100 * (1 - scores(y_te, p)[m] / best[m]),
                 "vs recency-weighted mean": 100 * (1 - scores(y_te, p)[m] / scores(y_te, test_pred[f'recency-weighted mean (alpha={ALPHA})'])[m])})
pl.DataFrame(rows)

metric,ours,best baseline,baseline,improvement_pct,vs recency-weighted mean
str,f64,str,f64,f64,f64
"""MAE""",11.471,"""Markov chain on price buckets: mean""",11.934,3.881,13.618
"""MSE""",254.705,"""Markov chain on price buckets: mean""",276.721,7.956,30.831
"""MSLE""",1.319,"""shrunk user mean (k=5)""",1.408,6.290,32.075


The three picks, chosen on validation: the absolute-loss GBM for MAE, and the squared-loss and log-loss GBMs blended with the hurdle variant for MSE and MSLE.

In [11]:
PICKS = {"MAE": ("GBM, absolute head", "p_plain_abs"), "MSE": ("GBM + hurdle, squared", "p_blend_sq"),
         "MSLE": ("GBM + hurdle, log", "p_blend_log")}
pl.DataFrame([{"metric": m, "model": name, "validation": scores(y_va, va[col].to_numpy())[m],
               "test": scores(y_te, te[col].to_numpy())[m]} for m, (name, col) in PICKS.items()])

metric,model,validation,test
str,str,f64,f64
"""MAE""","""GBM, absolute head""",11.589,11.471
"""MSE""","""GBM + hurdle, squared""",268.924,254.705
"""MSLE""","""GBM + hurdle, log""",1.436,1.319


Each pick wins its own metric, but a model that wins MSE is not automatically good on MSLE. This table scores every pick on all three metrics, next to the two constants that suit MSE and MSLE, so the trade-off is visible.

In [12]:
table({"MAE pick: GBM, absolute head": te["p_plain_abs"].to_numpy(),
       "MSE pick: GBM + hurdle, squared": te["p_blend_sq"].to_numpy(),
       "MSLE pick: GBM + hurdle, log": te["p_blend_log"].to_numpy(),
       "constant: median": test_pred["constant: median"],
       "constant: geometric mean": test_pred["constant: geometric mean"]}, y_te)

model,MAE,MSE,MSLE
str,f64,f64,f64
"""MAE pick: GBM, absolute head""",11.471,269.066,1.404
"""MSE pick: GBM + hurdle, squared""",11.619,254.705,1.510
"""MSLE pick: GBM + hurdle, log""",12.341,320.811,1.319
"""constant: median""",12.160,300.371,1.544
"""constant: geometric mean""",13.734,382.032,1.517


The squared-loss pick has the best MSE (254.7), but its MSLE is 1.510, barely below the geometric-mean constant at 1.517. The log pick is the best on MSLE (1.319), but its MSE is 320.8, worse than the median constant's 300.4. One model does not win both metrics, which is why we keep one per metric.

The MAE pick is the absolute-loss GBM. It has the best test MAE of the three picks (11.47), but its MSE (269.1) is above the MSE pick's 254.7, and its MSLE (1.404) is above the log pick's 1.319. The absolute loss targets the median, so it trades some squared and log error for a lower typical miss.

---

## Is the improvement real?

One user's reviews are not independent, so we resample users, 500 times with replacement, and recompute the gain each time. Improvement is baseline loss minus our loss, so positive means ours is better. Each interval is a 95% percentile interval from this bootstrap. All three gaps against the best baseline exclude zero: MAE by \$0.46, MSE by 22 and MSLE by 0.089.

The other rows check the hurdle variant against the same GBM without it. The intervals include zero (MSE -0.08, MSLE 0.000), so on test the hurdle neither helps nor hurts.

In [13]:
uid = te["user_id"].to_numpy()
_, user_code = np.unique(uid, return_inverse=True)
n_users = user_code.max() + 1
cnt = np.bincount(user_code, minlength=n_users)
draws = np.random.default_rng(SEED).multinomial(n_users, np.full(n_users, 1 / n_users), size=500)


def loss(metric, p):
    p = np.clip(p, 0, None)
    return {"MAE": np.abs(y_te - p), "MSE": (y_te - p) ** 2, "MSLE": (np.log1p(y_te) - np.log1p(p)) ** 2}[metric]

In [14]:
HEAD_OF = {"MAE": "abs", "MSE": "sq", "MSLE": "log"}
rows = []
for m, p in ours.items():
    best = test_table.filter(pl.col("model").is_in(baseline_names)).sort(m)["model"][0]
    refs = {best: test_pred[best]}
    plain = te[f"p_plain_{HEAD_OF[m]}"].to_numpy()
    if not np.array_equal(plain, p):
        refs["GBM, same head without hurdle"] = plain
    for ref, ref_pred in refs.items():
        diff = np.bincount(user_code, weights=loss(m, ref_pred) - loss(m, p), minlength=n_users)
        boot = (draws @ diff) / (draws @ cnt)
        rows.append({"metric": m, "compared with": ref, "improvement": (loss(m, ref_pred) - loss(m, p)).mean(),
                     "ci_low": np.quantile(boot, 0.025), "ci_high": np.quantile(boot, 0.975)})
pl.DataFrame(rows)

metric,compared with,improvement,ci_low,ci_high
str,str,f64,f64,f64
"""MAE""","""Markov chain on price buckets: mean""",0.463,0.435,0.493
"""MSE""","""Markov chain on price buckets: mean""",22.017,21.195,22.813
"""MSE""","""GBM, same head without hurdle""",-0.082,-0.233,0.050
"""MSLE""","""shrunk user mean (k=5)""",0.089,0.085,0.092
"""MSLE""","""GBM, same head without hurdle""",0.000,-0.001,0.001


In [15]:
pl.DataFrame([{"metric": m, "model": k, "validation": scores(y_va, va[f"p_{k}"].to_numpy())[m]}
              for m, ks in {"MSLE": ["plain_log", "hurdle_log", "blend_log"],
                            "MSE": ["plain_sq", "hurdle_sq", "blend_sq"]}.items() for k in ks])

metric,model,validation
str,str,f64
"""MSLE""","""plain_log""",1.439
"""MSLE""","""hurdle_log""",1.439
"""MSLE""","""blend_log""",1.436
"""MSE""","""plain_sq""",269.114
"""MSE""","""hurdle_sq""",269.654
"""MSE""","""blend_sq""",268.924


The table below gives a 95% interval for every model on test, using the same user bootstrap. The intervals are for each model's own loss, not for the gap to a baseline.

In [16]:
def mean_ci(pred):
    out = {}
    for m in ("MAE", "MSE", "MSLE"):
        l = loss(m, pred)
        boot = (draws @ np.bincount(user_code, weights=l, minlength=n_users)) / (draws @ cnt)
        out[m] = f"{l.mean():.3f} [{np.quantile(boot, 0.025):.3f}, {np.quantile(boot, 0.975):.3f}]"
    return out


pl.DataFrame([{"model": name, **mean_ci(pred)} for name, pred in test_pred.items()])

model,MAE,MSE,MSLE
str,str,str,str
"""constant: median""","""12.160 [12.072, 12.232]""","""300.371 [292.142, 311.935]""","""1.544 [1.528, 1.558]"""
"""constant: mean""","""12.189 [12.108, 12.253]""","""289.245 [281.221, 300.656]""","""1.596 [1.579, 1.611]"""
"""constant: geometric mean""","""13.734 [13.625, 13.828]""","""382.032 [372.924, 393.398]""","""1.517 [1.504, 1.528]"""
"""last price""","""14.917 [14.799, 15.019]""","""462.716 [442.703, 490.308]""","""2.458 [2.432, 2.482]"""
"""user median (last 10)""","""13.343 [13.241, 13.431]""","""377.780 [358.467, 402.719]""","""2.061 [2.037, 2.082]"""
"""recency-weighted mean (alpha=0.1)""","""13.279 [13.183, 13.365]""","""368.237 [348.742, 393.217]""","""1.942 [1.920, 1.962]"""
"""shrunk user mean (k=5)""","""13.179 [13.085, 13.265]""","""358.551 [349.405, 369.972]""","""1.408 [1.395, 1.417]"""
"""Markov chain on price buckets: median""","""11.974 [11.891, 12.045]""","""294.240 [285.606, 305.464]""","""1.458 [1.444, 1.472]"""
"""Markov chain on price buckets: mean""","""11.934 [11.861, 12.000]""","""276.721 [268.528, 288.055]""","""1.541 [1.525, 1.556]"""


---

## Where the model helps

By how much history the user has, our MSLE beats the recency-weighted mean by 47% for users with one earlier review, because that baseline is then just their one price. By 20+ reviews, the gap shrinks to 9%. Against the constant, it goes the other way: the lead grows from 10% to 20% as history grows.

Of the test targets, 60,514 (42%) come from users with no review before the test period. Their only history is earlier reviews inside the test period. They are harder (MSLE 1.461 against 1.215), but the model still beats both references on them.

In [17]:
seg = te.select("n", "user_id", "price").with_columns(
    history=pl.col("n").cut([1, 4, 19], labels=["1", "2-4", "5-19", "20+"]).cast(pl.Enum(["1", "2-4", "5-19", "20+"])),
    seen_before_test=pl.col("user_id").is_in(full["user_id"].implode()),
    model=pl.Series(ours["MSLE"]), ewa=pl.Series(test_pred[f"recency-weighted mean (alpha={ALPHA})"]),
    constant=pl.Series(test_pred["constant: geometric mean"]),
    shrunk=pl.Series(test_pred[f"shrunk user mean (k={K_SHRINK})"]))
sle = lambda c: (pl.col("price").log1p() - pl.col(c).clip(0).log1p()) ** 2
agg = lambda g: seg.group_by(g).agg(targets=pl.len(), MSLE_model=sle("model").mean(), MSLE_recency=sle("ewa").mean(),
                                    MSLE_constant=sle("constant").mean(), MSLE_shrunk=sle("shrunk").mean())
display(agg("history").sort("history"), agg("seen_before_test"))

history,targets,MSLE_model,MSLE_recency,MSLE_constant,MSLE_shrunk
enum,u32,f64,f64,f64,f64
"""1""",50275,1.481,2.789,1.640,1.580
"""2-4""",47793,1.352,1.743,1.552,1.447
"""5-19""",34474,1.154,1.263,1.392,1.231
"""20+""",10663,0.943,1.034,1.184,0.991


seen_before_test,targets,MSLE_model,MSLE_recency,MSLE_constant,MSLE_shrunk
bool,u32,f64,f64,f64,f64
false,60514,1.461,2.614,1.631,1.553
true,82691,1.215,1.450,1.434,1.301


Next we bootstrap each history segment on its own. Within a segment, we compare our MSLE pick with the shrunk user mean (the best MSLE baseline) and with the recency-weighted mean, using the same user resampling as before. The gain is the average MSLE saved per target in the segment, so positive means ours is better.

In [18]:
REFS = {f"shrunk user mean (k={K_SHRINK})": test_pred[f"shrunk user mean (k={K_SHRINK})"],
        f"recency-weighted mean (alpha={ALPHA})": test_pred[f"recency-weighted mean (alpha={ALPHA})"]}
groups = {"history": seg["history"].cast(pl.Utf8).to_numpy(), "seen before test": seg["seen_before_test"].to_numpy().astype(str)}
our_loss = loss("MSLE", ours["MSLE"])
rows = []
for ref, ref_pred in REFS.items():
    gap = loss("MSLE", ref_pred) - our_loss
    for col, values in groups.items():
        for v in np.unique(values):
            mask = values == v
            diff = np.bincount(user_code, weights=gap * mask, minlength=n_users)
            cnt_m = np.bincount(user_code, weights=mask.astype(float), minlength=n_users)
            boot = (draws @ diff) / (draws @ cnt_m)
            rows.append({"compared with": ref, "segment": f"{col} {v}", "targets": int(mask.sum()),
                         "MSLE gain": gap[mask].mean(), "ci_low": np.quantile(boot, 0.025),
                         "ci_high": np.quantile(boot, 0.975)})
seg_gain = pl.DataFrame(rows)
seg_gain

compared with,segment,targets,MSLE gain,ci_low,ci_high
str,str,i64,f64,f64,f64
"""shrunk user mean (k=5)""","""history 1""",50275,0.099,0.092,0.106
"""shrunk user mean (k=5)""","""history 2-4""",47793,0.095,0.088,0.100
"""shrunk user mean (k=5)""","""history 20+""",10663,0.048,0.037,0.063
"""shrunk user mean (k=5)""","""history 5-19""",34474,0.078,0.070,0.084
"""shrunk user mean (k=5)""","""seen before test False""",60514,0.092,0.085,0.099
"""shrunk user mean (k=5)""","""seen before test True""",82691,0.086,0.081,0.090
"""recency-weighted mean (alpha=0.1)""","""history 1""",50275,1.308,1.274,1.343
"""recency-weighted mean (alpha=0.1)""","""history 2-4""",47793,0.391,0.372,0.409
"""recency-weighted mean (alpha=0.1)""","""history 20+""",10663,0.092,0.068,0.122


In [19]:
import hvplot.pandas
import holoviews as hv

hist = seg_gain.filter(pl.col("compared with") == f"shrunk user mean (k={K_SHRINK})",
                       pl.col("segment").str.starts_with("history")).to_pandas()
hist["reviews before"] = hist["segment"].str.replace("history ", "", regex=False)
hist["text"] = hist["MSLE gain"].map("{:.2f}".format)
bars = hist.hvplot.bar(x="reviews before", y="MSLE gain", color="#4c78a8", width=640, height=320,
                       xlabel="reviews the user had before the target", ylabel="MSLE saved per target vs shrunk mean")
labels = hv.Labels(hist, kdims=["reviews before", "MSLE gain"], vdims=["text"])
bars * labels

:Overlay
   .Bars.I   :Bars   [reviews before]   (MSLE gain)
   .Labels.I :Labels   [reviews before,MSLE gain]   (text)

Against the shrunk mean, the MSLE gain is 0.099 for users with one earlier review, 0.095 for two to four, 0.078 for five to nineteen, and 0.048 for twenty or more. Every interval is far from zero. The gains against the recency-weighted mean are much larger for short histories (1.308 for one review) and shrink to 0.092 at twenty or more. The gain holds for users who had reviewed before the test period (0.086) and for those who had not (0.092).

---

## Where it fails

The MSLE pick's predictions sit in a narrow range. Free games get a median prediction of \$8.69, and \$30-60 games get \$13.57, when their true medians are \$0 and \$44.99. The MSLE pick has little signal for telling a \$60 game from a \$10 game before the review happens, so it guesses near the middle. The error is largest for free games (MSLE 4.99) and for \$60+ (5.18, only 212 targets), and smallest for \$5-10 (0.17), where predictions cluster near \$10.

Free games are 14.2% of test targets but 53.7% of the MSLE. The squared-error view flips that: \$30-60 games carry 56% of the MSE and \$60+ another 9%, so expensive games drive MSE.

In [20]:
seg = seg.with_columns(mse_model=pl.Series(ours["MSE"]))
total_se = float(((seg["price"] - seg["mse_model"]) ** 2).sum())
calib = (seg.with_columns(true_bucket=bucket_of("price"))
            .group_by("true_bucket").agg(targets=pl.len(), true_median=pl.col("price").median(),
                                         predicted_median=pl.col("model").median(), MSLE=sle("model").mean(),
                                         MSLE_shrunk=sle("shrunk").mean(),
                                         share_of_MSE=((pl.col("price") - pl.col("mse_model")) ** 2).sum() / total_se)
            .sort("true_bucket"))
calib

true_bucket,targets,true_median,predicted_median,MSLE,MSLE_shrunk,share_of_MSE
enum,u32,f64,f64,f64,f64,f64
"""free""",20309,0.000,8.688,4.994,4.776,0.164
"""$0-5""",10572,3.990,9.413,1.017,0.872,0.051
"""$5-10""",21489,9.990,10.879,0.168,0.116,0.053
"""$10-20""",49208,19.990,11.862,0.343,0.431,0.038
"""$20-30""",17765,29.990,13.168,0.736,1.000,0.044
"""$30-60""",23650,44.990,13.570,1.780,2.229,0.562
"""$60+""",212,79.990,11.828,5.181,5.621,0.088


In [21]:
import hvplot.pandas
import holoviews as hv

plot = calib.with_columns(pos=pl.int_range(pl.len()).cast(pl.Float64)).to_pandas()
plot["x_true"], plot["x_pred"] = plot["pos"] - 0.2, plot["pos"] + 0.2
plot["true_text"] = plot["true_median"].map("{:.0f}".format)
plot["pred_text"] = plot["predicted_median"].map("{:.0f}".format)
true_bars = plot.hvplot.bar(x="x_true", y="true_median", bar_width=0.38, color="#4c78a8", label="true median")
pred_bars = plot.hvplot.bar(x="x_pred", y="predicted_median", bar_width=0.38, color="#e45756", label="predicted median")
true_labels = hv.Labels(plot, kdims=["x_true", "true_median"], vdims=["true_text"])
pred_labels = hv.Labels(plot, kdims=["x_pred", "predicted_median"], vdims=["pred_text"])
ticks = list(zip(range(len(plot)), plot["true_bucket"].astype(str)))
(true_bars * pred_bars * true_labels * pred_labels).opts(xticks=ticks, width=640, height=300, ylabel="price ($)",
                             xlabel="true price bucket", legend_position="top_left")

:Overlay
   .Bars.True_median      :Bars   [x_true]   (true_median)
   .Bars.Predicted_median :Bars   [x_pred]   (predicted_median)
   .Labels.I              :Labels   [x_true,true_median]   (true_text)
   .Labels.II             :Labels   [x_pred,predicted_median]   (pred_text)

In [22]:
err = seg.with_columns(is_free=pl.col("price") == 0, error=sle("model"))
err.group_by("is_free").agg(share_of_targets=pl.len() / seg.height,
                            share_of_error=pl.col("error").sum() / err["error"].sum()).sort("is_free")

is_free,share_of_targets,share_of_error
bool,f64,f64
false,0.858,0.463
true,0.142,0.537


In [23]:
by_hist = (seg.group_by("history").agg(model=sle("model").mean(), recency=sle("ewa").mean(), constant=sle("constant").mean())
             .sort("history")
             .with_columns(gain_vs_recency_pct=(100 * (1 - pl.col("model") / pl.col("recency"))).round(0),
                           gain_vs_constant_pct=(100 * (1 - pl.col("model") / pl.col("constant"))).round(0)))
print(f"Cold-start share of test targets: {(~seg['seen_before_test']).mean():.0%}")
print(f"Share of MSE from $30 games and up: {calib.filter(pl.col('true_bucket').cast(pl.Utf8).is_in(['$30-60', '$60+']))['share_of_MSE'].sum():.0%}")
print(f"Share of MSLE from free games: {err.filter(pl.col('is_free'))['error'].sum() / err['error'].sum():.0%}")
by_hist

Cold-start share of test targets: 42%
Share of MSE from $30 games and up: 65%
Share of MSLE from free games: 54%


history,model,recency,constant,gain_vs_recency_pct,gain_vs_constant_pct
enum,f64,f64,f64,f64,f64
"""1""",1.481,2.789,1.640,47.000,10.000
"""2-4""",1.352,1.743,1.552,22.000,13.000
"""5-19""",1.154,1.263,1.392,9.000,17.000
"""20+""",0.943,1.034,1.184,9.000,20.000


---

## Summary

Each pick beats every baseline on its own metric. Against the best baseline for each metric, the gains are 4% to 8%, and against the recency-weighted mean they are 14% to 32%. The bootstrap over users keeps the gain over the best baseline well away from zero, and for the MSLE pick the gain holds in every history segment. The hurdle model made no measurable difference on test. The model has the lowest error with long histories, but its gain over the shrunk mean is smallest there (0.048 at 20+, against 0.099 at one review). It does worst on free games and on the most expensive ones.